In [12]:
import json
import asyncio
import re
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse, parse_qs

from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError
from bs4 import BeautifulSoup


# =========================
# Config (상담사례 114)
# =========================
BASE = "https://www.consumer.go.kr"

LIST_URL_TMPL = (
    "https://www.consumer.go.kr/user/ftc/consumer/cnsltcase/114/selectCnsltCaseList.do"
    "?page={page}&row=25&searchBgCode=&searchMdCode=&searchSmCode=&searchCnd=&searchWrd="
)

OUT_JSONL = "cnslt_cases_114_full.jsonl"
ERROR_JSONL = "cnslt_cases_114_errors.jsonl"

HEADLESS = True
TIMEOUT_MS = 30_000

MAX_RETRIES = 3
RETRY_BACKOFF_SEC = 1.5
CHECKPOINT_EVERY = 1

# selectors
LIST_ROW_SELECTOR = "table.tbl.col.data tbody tr"
LIST_LINK_SELECTOR = "td.title a"
DETAIL_TABLE_SELECTOR = "table.tbl.row.data"

# 병렬 상세 처리 개수(안전)
DETAIL_CONCURRENCY = 3


# =========================
# Utils
# =========================
def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def extract_case_sn(url: str) -> str | None:
    qs = parse_qs(urlparse(url).query)
    v = qs.get("prgnCnsltCaseSn")
    return v[0] if v else None

def make_doc_id(url: str) -> str:
    sn = extract_case_sn(url)
    return str(sn) if sn else f"url:{url}"

def normalize_text(text: str) -> str:
    """
    RAG용 '보수적' 정규화:
    - 연속 줄바꿈(3개 이상)만 2개로 축소
    - 탭/과한 공백 정리
    """
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def safe_text(el) -> str:
    if not el:
        return ""
    return normalize_text(el.get_text("\n", strip=True))


# =========================
# Parsing: 상세
# =========================
def parse_detail_html(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    def get_row(label: str) -> str:
        for th in soup.find_all("th"):
            if label in th.get_text(strip=True):
                td = th.find_next_sibling("td")
                if not td:
                    return ""
                div = td.select_one("div.bbs_view_content")
                return safe_text(div) if div else safe_text(td)
        return ""

    title = get_row("제목")
    source = get_row("출처")
    category = get_row("분류")
    question = get_row("질문")
    answer = get_row("답변")

    # RAG용 content 구성(사람이 읽는 형태 유지)
    parts = []
    if title: parts.append(f"제목: {title}")
    if category: parts.append(f"분류: {category}")
    if source: parts.append(f"출처: {source}")
    if question: parts.append(f"질문:\n{question}")
    if answer: parts.append(f"답변:\n{answer}")

    content = "\n\n".join(parts).strip()

    return {
        "id": make_doc_id(url),
        "url": url,
        "title": title,
        "source": source,
        "category": category,
        "question": question,
        "answer": answer,
        "content": content,
        "collected_at": now_iso(),
        "metadata": {
            "site": "consumer.go.kr",
            "doc_type": "consumer_counsel_case",
            "case_sn": extract_case_sn(url),
        },
    }


# =========================
# IO
# =========================
def load_seen_ids(path: str) -> set[str]:
    seen = set()
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    _id = obj.get("id")
                    if _id:
                        seen.add(str(_id))
                except:
                    pass
    except FileNotFoundError:
        pass
    return seen

def append_jsonl(fp, obj: dict):
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")
    fp.flush()


# =========================
# Network optimization
# =========================
async def block_heavy_assets(route):
    rtype = route.request.resource_type
    # 서버 렌더링 기반이라면 stylesheet 차단해도 보통 문제 없음
    if rtype in ("image", "font", "media", "stylesheet"):
        await route.abort()
    else:
        await route.continue_()


# =========================
# 목록 메타 추출 (중요!)
# =========================
async def extract_list_items(list_page) -> list[dict]:
    """
    목록의 tr 단위로:
    - link(상세 href)
    - 번호, 상담분야, 품목, 출처, 조회수
    를 한 번에 뽑아낸다.
    """
    # DOM에서 한 번에 뽑아오는 게 가장 안정적(HTML 파싱보다 빠름)
    items = await list_page.locator(LIST_ROW_SELECTOR).evaluate_all(
        """rows => rows.map(tr => {
            const get = (sel) => {
                const el = tr.querySelector(sel);
                return el ? el.textContent.trim() : "";
            };

            const a = tr.querySelector("td.title a");
            const href = a ? a.getAttribute("href") : "";

            return {
                href,
                no: get('td[aria-label="번호"]'),
                field: get('td[aria-label="상담분야"]'),
                item: get('td[aria-label="품목"]'),
                source_list: get('td[aria-label="출처"]'),
                views: get('td[aria-label="조회수"]'),
            };
        })"""
    )
    # href 정리 + 빈값 제외
    cleaned = []
    for it in items:
        href = it.get("href") or ""
        if not href:
            continue
        it["url"] = urljoin(BASE, href)
        it["case_sn"] = extract_case_sn(it["url"])
        cleaned.append(it)
    return cleaned


# =========================
# Main
# =========================
async def main(start_page=1, end_page=50):
    seen = load_seen_ids(OUT_JSONL)
    print("seen already:", len(seen))

    sem = asyncio.Semaphore(DETAIL_CONCURRENCY)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
            )
        )
        context.set_default_timeout(TIMEOUT_MS)
        await context.route("**/*", block_heavy_assets)

        list_page = await context.new_page()

        saved = 0
        skipped = 0
        failed = 0

        with open(OUT_JSONL, "a", encoding="utf-8") as out_fp, \
             open(ERROR_JSONL, "a", encoding="utf-8") as err_fp:

            async def fetch_detail(url: str, page_no: int, list_meta: dict):
                nonlocal saved, skipped, failed

                async with sem:
                    doc_id = make_doc_id(url)
                    if doc_id in seen:
                        skipped += 1
                        return

                    detail_page = await context.new_page()
                    try:
                        last_err = None
                        for attempt in range(1, MAX_RETRIES + 1):
                            try:
                                await detail_page.goto(url, wait_until="domcontentloaded")
                                await detail_page.wait_for_selector(DETAIL_TABLE_SELECTOR)

                                html = await detail_page.content()
                                item = parse_detail_html(html, url)

                                # 최소 검증
                                if not (item["title"] or item["question"] or item["answer"]):
                                    raise RuntimeError("empty parsed fields")

                                # ✅ 목록 메타 병합(1번 코드 장점 흡수)
                                item["list_meta"] = {
                                    "list_page": page_no,
                                    "no": list_meta.get("no", ""),
                                    "field": list_meta.get("field", ""),          # 상담분야
                                    "item": list_meta.get("item", ""),            # 품목
                                    "source_list": list_meta.get("source_list", ""),
                                    "views": list_meta.get("views", ""),
                                }

                                # ✅ metadata에도 필터 가능한 값들을 넣어두면 추후 편함
                                item["metadata"].update({
                                    "field": list_meta.get("field", ""),
                                    "item": list_meta.get("item", ""),
                                    "views": list_meta.get("views", ""),
                                    "source_list": list_meta.get("source_list", ""),
                                })

                                append_jsonl(out_fp, item)
                                seen.add(item["id"])
                                saved += 1
                                return

                            except Exception as e:
                                last_err = e
                                if attempt < MAX_RETRIES:
                                    await asyncio.sleep(RETRY_BACKOFF_SEC * attempt)
                                else:
                                    failed += 1
                                    append_jsonl(err_fp, {
                                        "url": url,
                                        "id": doc_id,
                                        "page": page_no,
                                        "error": repr(last_err),
                                        "at": now_iso(),
                                    })
                    finally:
                        await detail_page.close()

            for pg in range(start_page, end_page + 1):
                list_url = LIST_URL_TMPL.format(page=pg)

                # 목록 페이지도 재시도(links=0/timeout 대응)
                list_items = []
                for attempt in range(1, MAX_RETRIES + 1):
                    try:
                        await list_page.goto(list_url, wait_until="domcontentloaded")
                        await list_page.wait_for_selector(LIST_ROW_SELECTOR)
                        list_items = await extract_list_items(list_page)

                        # 링크가 0개면 "빈 목록/차단/일시 오류" 가능 → 재시도
                        if len(list_items) == 0:
                            raise RuntimeError("list empty (0 rows with href)")

                        break

                    except (PWTimeoutError, Exception) as e:
                        if attempt < MAX_RETRIES:
                            await asyncio.sleep(RETRY_BACKOFF_SEC * attempt)
                        else:
                            tr_cnt = await list_page.locator("tbody tr").count()
                            print(f"[list][fail] page={pg} tr={tr_cnt} err={repr(e)}")
                            append_jsonl(err_fp, {
                                "url": list_url,
                                "page": pg,
                                "error": f"list_page_failed: {repr(e)}",
                                "at": now_iso(),
                            })
                            list_items = []

                if not list_items:
                    continue

                print(f"[list] page={pg} items={len(list_items)}")

                tasks = []
                for meta in list_items:
                    url = meta["url"]
                    tasks.append(fetch_detail(url, pg, meta))

                await asyncio.gather(*tasks)

                if pg % CHECKPOINT_EVERY == 0:
                    print(f"✅ checkpoint page {pg} | saved={saved} skipped={skipped} failed={failed}")

        await browser.close()

    print("\n==== DONE ====")
    print(f"pages: {start_page}~{end_page}")
    print(f"saved: {saved}")
    print(f"skipped(seen): {skipped}")
    print(f"failed: {failed}")


# 실행 예시
# await main(start_page=1, end_page=100)
# await main(start_page=101, end_page=200)
# await main(start_page=201, end_page=300)
# await main(start_page=301, end_page=400)
# await main(start_page=401, end_page=500)
await main(start_page=501, end_page=600)

seen already: 11342
[list] page=501 items=17
✅ checkpoint page 501 | saved=0 skipped=17 failed=0
[list] page=502 items=17
✅ checkpoint page 502 | saved=0 skipped=34 failed=0
[list] page=503 items=17
✅ checkpoint page 503 | saved=0 skipped=51 failed=0
[list] page=504 items=17
✅ checkpoint page 504 | saved=0 skipped=68 failed=0
[list] page=505 items=17
✅ checkpoint page 505 | saved=0 skipped=85 failed=0
[list] page=506 items=17
✅ checkpoint page 506 | saved=0 skipped=102 failed=0
[list] page=507 items=17
✅ checkpoint page 507 | saved=0 skipped=119 failed=0
[list] page=508 items=17
✅ checkpoint page 508 | saved=0 skipped=136 failed=0
[list] page=509 items=17
✅ checkpoint page 509 | saved=0 skipped=153 failed=0
[list] page=510 items=17
✅ checkpoint page 510 | saved=0 skipped=170 failed=0
[list] page=511 items=17
✅ checkpoint page 511 | saved=0 skipped=187 failed=0
[list] page=512 items=17
✅ checkpoint page 512 | saved=0 skipped=204 failed=0
[list] page=513 items=17
✅ checkpoint page 513 | 

In [6]:
import json

INPUT = "cnslt_cases_full.jsonl"
OUTPUT = "cnslt_cases_full.json"

data = []
with open(INPUT, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"✅ 변환 완료: {OUTPUT} (총 {len(data)}건)")


✅ 변환 완료: cnslt_cases_full.json (총 11340건)
